In [3]:
import os
os.makedirs("/content/ews/report", exist_ok=True)
!unzip -o -q /content/ews_min.zip -d /content/ews
%cd /content/ews
!pip -q install polars pandas pyarrow psutil pyspark

print("\n--- KIỂM TRA MÔI TRƯỜNG VÀ DỮ LIỆU ---")
!ls -lh data_processed/bench/
!java -version
!python -c "import polars; print('Colab Polars version:', polars.__version__)"

/content/ews

--- KIỂM TRA MÔI TRƯỜNG VÀ DỮ LIỆU ---
total 177M
-rw-rw-rw- 1 root root  26M Nov 15 10:04 ytd_input_x1.parquet
-rw-rw-rw- 1 root root  51M Nov 15 10:04 ytd_input_x2.parquet
-rw-rw-rw- 1 root root 101M Nov 15 10:04 ytd_input_x4.parquet
[0.013s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.013s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
Colab Polars version: 1.35.2


In [4]:
%cd /content/ews

# 1. Đo x1 với 8 shuffle partitions
print("--- 1. Spark x1 (shuffle=8) ---")
!python src/06_spark_pipeline.py --input data_processed/bench/ytd_input_x1.parquet --label x1_shuf8 --shuffle-partitions 8 --repeats 3

# 2. Đo x1 với 8 shuffle partitions và tắt AQE
print("\n--- 2. Spark x1 (shuffle=8, AQE off) ---")
!python src/06_spark_pipeline.py --input data_processed/bench/ytd_input_x1.parquet --label x1_shuf8_aqeoff --shuffle-partitions 8 --aqe false --repeats 3

# 3. Đo x4 với 8 shuffle partitions
print("\n--- 3. Spark x4 (shuffle=8) ---")
!python src/06_spark_pipeline.py --input data_processed/bench/ytd_input_x4.parquet --label x4_shuf8 --shuffle-partitions 8 --repeats 3

/content/ews
--- 1. Spark x1 (shuffle=8) ---
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 15:08:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your

In [5]:
%cd /content/ews

# 1. Chạy x1 mặc định (để ghi file JSON có đủ thông số Java, Broadcast, AQE chuẩn)
print("--- 1. Spark x1 mặc định ---")
!python src/06_spark_pipeline.py --input data_processed/bench/ytd_input_x1.parquet --label x1_default --repeats 5

# 2. Đo lại pandas vs Polars (7 lần lặp, chuẩn đo ru_maxrss)
print("\n--- 2. Benchmark pandas vs Polars trên Colab (7 lần lặp) ---")
!python src/05_polars_pandas_benchmark.py --engines pandas polars_lazy --repeats 7

# 3. Nén toàn bộ các file kết quả (.json, .txt) và tự động tải về máy
!zip -q colab_results.zip spark_T1_result_*.json spark_plan_T1_*.txt report/*.csv
print("\nĐÃ XONG HẾT! Đang tải file colab_results.zip về máy...")
from google.colab import files
files.download('colab_results.zip')

/content/ews
--- 1. Spark x1 mặc định ---
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 15:14:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your pl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
!python -V
!pip show pandas


Python 3.13.18
Name: pandas
Version: 2.2.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
All rights reserved.

Copyright (c) 2011-2023, Open source contributors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyright holder nor the names of its
  contri

In [7]:
%cd /content/ews
!python src/06_spark_pipeline.py --input data_processed/bench/ytd_input_x2.parquet --label x2_default --repeats 5
!python src/06_spark_pipeline.py --input data_processed/bench/ytd_input_x4.parquet --label x4_default --repeats 5

/content/ews
[0.003s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 15:39:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java 